# CƠM QUÊ — Random Forest dự báo số đơn trong 7 ngày

Notebook độc lập cho mục 3.6 của đề cương: kiểm tra CSV → tạo đặc trưng lịch sử →
huấn luyện/đánh giá → biểu đồ → học lại toàn bộ → dự báo 7 ngày → tải kết quả.

**Cách dùng:** mở trên Google Colab bằng **File → Upload notebook**, chọn Python 3 (>= 3.12),
chỉnh ô cấu hình rồi **Runtime → Run all**. CPU đủ để chạy; không cần GPU, database hoặc API key.
Mã nguồn xử lý đã được nhúng trong notebook, không cần clone repo. Chạy lại từ bước 2 tạo thư mục run mới.

**Mặc định là dữ liệu mô phỏng, seed 42. Kết quả không chứng minh chất lượng trên bán hàng thực tế.**
Nguồn `application_export` cũng có thể là đơn nhập thử; cần xác minh nguồn trước khi kết luận hiệu quả kinh doanh.
Notebook không tự thay đổi bộ kết quả Dashboard hay triển khai lên Render.

## 1. Chọn dữ liệu và ngày tham chiếu

- `DATA_MODE="demo"`, `DATA_SOURCE="mo_phong"`: tự sinh chuỗi ngày liên tục đến hết hôm trước ngày tham chiếu.
- `DATA_MODE="csv"`: tải CSV lên Colab (để trống `CSV_PATH`) hoặc nhập đường dẫn file đã có.
  Chọn đúng `DATA_SOURCE`: `mo_phong` cho CSV demo; `application_export` cho CSV xuất từ ứng dụng.
- `REFERENCE_DATE=""`: ngày hiện tại theo `Asia/Ho_Chi_Minh`.
  Có thể đặt `2026-12-12` để kiểm thử; khi đó đầu vào phải kết thúc ngày `2026-12-11`,
  dự báo cho `12/12–18/12/2026`. Với CSV lịch sử, nhập ngày ngay sau ngày cuối CSV.
- CSV phải có `ngay,so_don,nguon_du_lieu`, ít nhất **120 ngày liên tục**, số đơn nguyên không âm.
  Ngày theo `YYYY-MM-DD`, không thiếu/trùng ngày hoặc trộn hai nguồn. Không tự bù dữ liệu ứng dụng bằng mô phỏng.
- Những dòng từ ngày tham chiếu trở đi không được đưa vào snapshot huấn luyện.

In [ ]:
#@title Cấu hình dữ liệu
DATA_MODE = "demo" #@param ["demo", "csv"]
DATA_SOURCE = "mo_phong" #@param ["mo_phong", "application_export"]
CSV_PATH = "" #@param {type:"string"}
REFERENCE_DATE = "" #@param {type:"string"}
DOWNLOAD_RESULTS = True #@param {type:"boolean"}

## 2. Chuẩn bị thư viện và mã nguồn

Các thư viện huấn luyện dùng phiên bản cố định trong `requirements-forecast.txt`,
khớp môi trường AI hiện tại của dự án. Chỉ cài khi thiếu hoặc khác phiên bản.
Huấn luyện và vẽ biểu đồ chạy trong tiến trình Python mới, tránh dùng lại NumPy/pandas đã được kernel Colab nạp trước khi cài.
Lỗi cài đặt/huấn luyện sẽ được giữ lại để xử lý, không tự đổi mô hình hoặc nguồn dữ liệu.

In [ ]:
#@title Chuẩn bị môi trường và mã xử lý (mở ô này nếu cần xem mã nhúng)
import csv
from html import escape
from importlib import metadata
import json
import os
from pathlib import Path
import subprocess
import sys
from uuid import uuid4

if sys.version_info < (3, 12):
    raise RuntimeError("Notebook cần Python >= 3.12. Hãy chọn runtime Python tương thích.")
TRAINING_REQUIREMENTS = {'numpy': '2.5.4', 'pandas': '3.1.0', 'scikit-learn': '1.9.1', 'joblib': '1.6.0', 'matplotlib': '3.11.2', 'tzdata': '2026.6'}
def ensure_training_packages(requirements):
    missing = []
    for package, version in requirements.items():
        try:
            installed = metadata.version(package)
        except metadata.PackageNotFoundError:
            installed = None
        if installed != version:
            missing.append(f"{package}=={version}")
    if missing:
        print("Đang cài thư viện. Bước này có thể mất vài phút ở lần đầu...", flush=True)
        process = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing],
                                 capture_output=True, text=True, encoding="utf-8", errors="replace")
        if process.returncode:
            raise RuntimeError("Chưa cài được thư viện. Chi tiết lỗi đầu tiên:\n" + process.stdout + process.stderr)

ensure_training_packages(TRAINING_REQUIREMENTS)
try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None
try:
    from IPython.display import display, HTML, Image
except ImportError:
    display, HTML, Image = print, None, None

WORK_ROOT = Path(os.environ.get("QLNH_FORECAST_NOTEBOOK_ROOT", "/content/qlnh_order_forecast" if colab_files else "artifacts/ai/forecast_notebook")).resolve()
SUPPORT_ROOT = WORK_ROOT / ("source_" + uuid4().hex[:12])
PACKAGE = SUPPORT_ROOT / "ai_tools"
PACKAGE.mkdir(parents=True, exist_ok=False)
SOURCE_FILES = {'__init__.py': '', 'forecast_store.py': '"""Forecast data/publication contract. Standard library only; also used by Django."""\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport re\nimport time\nimport uuid\nfrom datetime import date, datetime, timedelta\nfrom pathlib import Path\nfrom zoneinfo import ZoneInfo\n\nVIETNAM = ZoneInfo("Asia/Ho_Chi_Minh")\nSOURCES = ("mo_phong", "application_export")\nFILES = ("input.csv", "du_bao_7_ngay.csv", "evaluation.json",\n         "kiem_tra_du_bao.csv", "forecast_demo.joblib")\n\n\ndef vietnam_today():\n    return datetime.now(VIETNAM).date()\n\n\ndef source_directory(root, source):\n    if source not in SOURCES:\n        raise ValueError("Unknown forecast source")\n    return Path(root) / source\n\n\ndef _retry_windows_sharing(operation):\n    # Windows may briefly deny opening/replacing a file while a previous handle\n    # is closing. Retry only this transient sharing case, never expose a partial file.\n    for attempt in range(10):\n        try:\n            return operation()\n        except PermissionError:\n            if os.name != "nt" or attempt == 9:\n                raise\n            time.sleep(min(0.01 * (attempt + 1), 0.05))\n\n\ndef read_bytes(path):\n    return _retry_windows_sharing(Path(path).read_bytes)\n\n\ndef atomic_bytes(path, content):\n    """Replace one file on the same filesystem, never expose partial contents."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")\n    try:\n        with temporary.open("xb") as stream:\n            stream.write(content)\n            stream.flush()\n            os.fsync(stream.fileno())\n        _retry_windows_sharing(lambda: os.replace(temporary, path))\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef atomic_json(path, value):\n    atomic_bytes(path, json.dumps(value, ensure_ascii=False, indent=2,\n                                 allow_nan=False).encode("utf-8"))\n\n\ndef csv_bytes(rows, fields):\n    buffer = io.StringIO(newline="")\n    writer = csv.DictWriter(buffer, fieldnames=fields)\n    writer.writeheader()\n    writer.writerows(rows)\n    return buffer.getvalue().encode("utf-8-sig")\n\n\ndef read_daily(path, source):\n    source_directory(".", source)\n    with Path(path).open(encoding="utf-8-sig", newline="") as stream:\n        reader = csv.DictReader(stream)\n        if not {"ngay", "so_don", "nguon_du_lieu"}.issubset(reader.fieldnames or []):\n            raise ValueError("Daily CSV missing required columns")\n        rows = list(reader)\n    previous = None\n    for row in rows:\n        day = date.fromisoformat(row["ngay"])\n        if previous is not None and day != previous + timedelta(days=1):\n            raise ValueError("Daily CSV must have consecutive unique ordered dates; no filling gaps")\n        if row["nguon_du_lieu"] != source:\n            raise ValueError("Daily CSV source mismatch; sources must never be mixed")\n        if not re.fullmatch(r"[0-9]+", row["so_don"]):\n            raise ValueError("Orders must be nonnegative integers")\n        previous = day\n    if not rows:\n        raise ValueError("Daily CSV is empty")\n    return rows\n\n\ndef training_rows(path, source, reference_date):\n    # A snapshot for a historical reference date must never contain future data.\n    rows = [row for row in read_daily(path, source)\n            if date.fromisoformat(row["ngay"]) < reference_date]\n    if len(rows) < 120:\n        raise ValueError("At least 120 consecutive training days are required")\n    if date.fromisoformat(rows[-1]["ngay"]) != reference_date - timedelta(days=1):\n        raise ValueError("Training data must end exactly yesterday; no synthetic application fallback")\n    return rows\n\n\ndef sha256(content):\n    return hashlib.sha256(content).hexdigest()\n\n\ndef validate_result(metadata, rows, source, reference_date, run_id):\n    if metadata["schema_version"] != 2 or metadata["run_id"] != run_id:\n        raise ValueError("Metadata run identity mismatch")\n    if metadata["data_source"] != source or metadata["timezone"] != "Asia/Ho_Chi_Minh":\n        raise ValueError("Metadata source/timezone mismatch")\n    if metadata["reference_date"] != reference_date.isoformat():\n        raise ValueError("Reference date mismatch")\n    if metadata["training_last_date"] != (reference_date - timedelta(days=1)).isoformat():\n        raise ValueError("Training cutoff mismatch")\n    if metadata["horizon_days"] != 7 or metadata["evaluation_days"] != 28:\n        raise ValueError("Invalid forecast/evaluation horizon")\n    if not metadata["final_model_refitted_on_all_days"]:\n        raise ValueError("Final model was not refitted")\n    for key in ("mae_ai", "mae_baseline", "rmse_ai"):\n        value = metadata[key]\n        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value < 0:\n            raise ValueError("Invalid evaluation metric")\n    expected = [(reference_date + timedelta(days=i)).isoformat() for i in range(7)]\n    if [row["ngay"] for row in rows] != expected:\n        raise ValueError("Forecast must have exactly 7 consecutive days starting on reference date")\n    for row in rows:\n        if row["nguon_du_lieu"] != source or row["run_id"] != run_id:\n            raise ValueError("Forecast CSV run/source mismatch")\n        if not re.fullmatch(r"[0-9]+", row["so_don_du_kien"]):\n            raise ValueError("Predictions must be nonnegative integers")\n\n\ndef load_bundle(folder, source, reference_date, run_id, hashes=None, full=False):\n    folder = Path(folder)\n    names = FILES if full else ("evaluation.json", "du_bao_7_ngay.csv")\n    contents = {}\n    for name in names:\n        content = read_bytes(folder / name)\n        if not content:\n            raise ValueError(f"Empty artifact: {name}")\n        if hashes is not None and sha256(content) != hashes[name]:\n            raise ValueError(f"Artifact checksum mismatch: {name}")\n        contents[name] = content\n    metadata = json.loads(contents["evaluation.json"])\n    rows = list(csv.DictReader(io.StringIO(contents["du_bao_7_ngay.csv"].decode("utf-8-sig"))))\n    validate_result(metadata, rows, source, reference_date, run_id)\n    if full:\n        if metadata["input_sha256"] != sha256(contents["input.csv"]):\n            raise ValueError("Training input checksum mismatch")\n        if metadata["model_sha256"] != sha256(contents["forecast_demo.joblib"]):\n            raise ValueError("Model/metadata checksum mismatch")\n        daily = training_rows(folder / "input.csv", source, reference_date)\n        if metadata["training_days"] != len(daily):\n            raise ValueError("Training input length mismatch")\n        evaluation = list(csv.DictReader(io.StringIO(contents["kiem_tra_du_bao.csv"].decode("utf-8-sig"))))\n        expected_days = [(reference_date - timedelta(days=i)).isoformat() for i in range(28, 0, -1)]\n        if [row["ngay"] for row in evaluation] != expected_days:\n            raise ValueError("Evaluation must cover the last 28 training days")\n        for i, row in enumerate(evaluation):\n            if row["run_id"] != run_id or row["nguon_du_lieu"] != source:\n                raise ValueError("Evaluation run/source mismatch")\n            origin = expected_days[(i // 7) * 7]\n            if row["ngay_bat_dau_du_bao"] != origin or int(row["buoc_du_bao"]) != i % 7 + 1:\n                raise ValueError("Invalid rolling evaluation origin")\n            for key in ("thuc_te", "du_bao_ai", "du_bao_tuan_truoc"):\n                value = float(row[key])\n                if not math.isfinite(value) or value < 0:\n                    raise ValueError("Invalid evaluation prediction")\n    return metadata, rows\n\n\ndef publish(root, source, run_id, reference_date):\n    directory = source_directory(root, source)\n    folder = directory / "runs" / run_id\n    load_bundle(folder, source, reference_date, run_id, full=True)\n    manifest = {\n        "schema_version": 2, "run_id": run_id, "data_source": source,\n        "reference_date": reference_date.isoformat(),\n        "published_at": datetime.now(VIETNAM).isoformat(),\n        "sha256": {name: sha256((folder / name).read_bytes()) for name in FILES},\n    }\n    # Only this pointer changes. Published run directories must never be edited/deleted.\n    atomic_json(directory / "current.json", manifest)\n    return manifest\n\n\ndef read_published(root, source):\n    directory = source_directory(root, source)\n    manifest = json.loads(read_bytes(directory / "current.json"))\n    run_id = manifest["run_id"]\n    if not isinstance(run_id, str) or not re.fullmatch(r"[A-Za-z0-9_-]{1,100}", run_id):\n        raise ValueError("Invalid run directory")\n    if manifest["schema_version"] != 2 or manifest["data_source"] != source:\n        raise ValueError("Invalid publication manifest")\n    reference_date = date.fromisoformat(manifest["reference_date"])\n    metadata, rows = load_bundle(directory / "runs" / run_id, source,\n                                 reference_date, run_id, manifest["sha256"])\n    return manifest, metadata, rows\n\n\nclass UpdateAlreadyRunning(RuntimeError):\n    pass\n\n\nclass UpdateLock:\n    """OS lock; released even on process death. Never unlink the lock file."""\n    def __init__(self, root):\n        self.path = Path(root) / "update.lock"\n\n    def __enter__(self):\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.stream = self.path.open("a+b")\n        self.stream.seek(0, os.SEEK_END)\n        if self.stream.tell() == 0:\n            self.stream.write(b"0")\n            self.stream.flush()\n        self.stream.seek(0)\n        try:\n            if os.name == "nt":\n                import msvcrt\n                msvcrt.locking(self.stream.fileno(), msvcrt.LK_NBLCK, 1)\n            else:\n                import fcntl\n                fcntl.flock(self.stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except OSError as exc:\n            self.stream.close()\n            raise UpdateAlreadyRunning("Another forecast update holds the lock") from exc\n        return self\n\n    def __exit__(self, *args):\n        if os.name == "nt":\n            import msvcrt\n            self.stream.seek(0)\n            msvcrt.locking(self.stream.fileno(), msvcrt.LK_UNLCK, 1)\n        else:\n            import fcntl\n            fcntl.flock(self.stream.fileno(), fcntl.LOCK_UN)\n        self.stream.close()\n', 'forecast_demo.py': '"""Append-only seeded demo series, compatible with the original seed-42 CSV."""\nimport math\nimport random\nfrom datetime import date, timedelta\nfrom pathlib import Path\n\nfrom .forecast_store import atomic_bytes, csv_bytes, read_daily\n\nSTART = date(2025, 10, 30)\nSEED = 42\nFIELDS = ["ngay", "so_don", "nguon_du_lieu"]\n\n\ndef extend_demo(path, through):\n    """Caller holds the update lock. Preserve old rows, replay RNG before appending."""\n    path = Path(path)\n    rows = read_daily(path, "mo_phong") if path.exists() else []\n    if rows and rows[0]["ngay"] != START.isoformat():\n        raise ValueError("Demo start/seed contract differs from original series")\n    existing_end = date.fromisoformat(rows[-1]["ngay"]) if rows else START - timedelta(days=1)\n    if existing_end >= through:\n        return rows\n    rng = random.Random(SEED)\n    factors = [0.85, 0.80, 0.90, 0.95, 1.15, 1.50, 1.35]\n    for i in range((through - START).days + 1):\n        day = START + timedelta(days=i)\n        noise = rng.gauss(0, 7)\n        if day <= existing_end:\n            continue\n        expected = (40 + 0.025 * i) * factors[day.weekday()] * (1 + 0.10 * math.sin(2 * math.pi * i / 365))\n        rows.append({"ngay": day.isoformat(), "so_don": str(max(0, round(expected + noise))),\n                     "nguon_du_lieu": "mo_phong"})\n    atomic_bytes(path, csv_bytes(rows, FIELDS))\n    return rows\n', 'forecast_training.py': 'import json\nfrom datetime import date, timedelta\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error\n\nfrom ai_tools.forecast_store import training_rows, sha256\n\nSEED = 42\nHORIZON = 7\nTEST_DAYS = 28\n\nFEATURES = [\n    "thu", "cuoi_tuan", "chi_so_ngay",\n    "don_hom_qua", "don_7_ngay_truoc",\n    "don_14_ngay_truoc", "don_28_ngay_truoc",\n    "trung_binh_7_ngay", "trung_binh_28_ngay",\n]\n\ndef feature_row(day, history, first_day):\n    if len(history) < 28:\n        raise ValueError("Can 28 ngay lich su.")\n    return [\n        day.weekday(),\n        int(day.weekday() >= 5),\n        (day - first_day).days,\n        history[-1],\n        history[-7],\n        history[-14],\n        history[-28],\n        float(np.mean(history[-7:])),\n        float(np.mean(history[-28:])),\n    ]\n\n\ndef fit_model(values, known_dates, first_day):\n    # Moi dong chi dung lich su truoc ngay can du bao.\n    X = [\n        feature_row(known_dates[i], values[:i], first_day)\n        for i in range(28, len(values))\n    ]\n    model = RandomForestRegressor(\n        n_estimators=200,\n        max_depth=8,\n        min_samples_leaf=3,\n        random_state=SEED,\n        n_jobs=-1,\n    )\n    model.fit(pd.DataFrame(X, columns=FEATURES), values[28:])\n    return model\n\n\ndef forecast(model, values, next_day, first_day, days=7):\n    history = list(map(float, values))\n    predictions = []\n    for step in range(days):\n        day = next_day + timedelta(days=step)\n        X = pd.DataFrame(\n            [feature_row(day, history, first_day)],\n            columns=FEATURES,\n        )\n        pred = max(0.0, float(model.predict(X)[0]))\n        predictions.append(pred)\n        # Ngay tuong lai chua co thuc te: dung du bao vua tao.\n        history.append(pred)\n    return predictions\n\n\ndef baseline_forecast(values, days=7):\n    history = list(map(float, values))\n    predictions = []\n    for _ in range(days):\n        pred = history[-7]\n        predictions.append(pred)\n        history.append(pred)\n    return predictions\n\n\ndef train(input_path, output, source, reference_date, run_id):\n    rows = training_rows(input_path, source, reference_date)\n    dates = [date.fromisoformat(row["ngay"]) for row in rows]\n    y = np.array([int(row["so_don"]) for row in rows], dtype=float)\n    first_day = dates[0]\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    # Updater created this unique run directory and immutable input snapshot.\n    if any((output / name).exists() for name in ("evaluation.json", "forecast_demo.joblib", "du_bao_7_ngay.csv", "kiem_tra_du_bao.csv")):\n        raise FileExistsError("Refuse to overwrite an existing training result")\n    print("1. DU LIEU:", len(y), "ngay")\n    print("2. DANH GIA 4 LUOT, MOI LUOT 7 NGAY")\n\n    evaluation_rows = []\n    test_start = len(y) - TEST_DAYS\n\n    for origin in range(test_start, len(y), HORIZON):\n        # Moi luot chi duoc hoc tu cac ngay da xay ra.\n        train_y = y[:origin]\n        model = fit_model(train_y, dates[:origin], first_day)\n        model_preds = forecast(model, train_y, dates[origin], first_day, HORIZON)\n        base_preds = baseline_forecast(train_y, HORIZON)\n\n        for h in range(HORIZON):\n            evaluation_rows.append({\n                "run_id": run_id, "nguon_du_lieu": source,\n                "ngay_bat_dau_du_bao": dates[origin].isoformat(),\n                "ngay": dates[origin + h].isoformat(),\n                "buoc_du_bao": h + 1,\n                "thuc_te": int(y[origin + h]),\n                "du_bao_ai": model_preds[h],\n                "du_bao_tuan_truoc": base_preds[h],\n            })\n\n    evaluation = pd.DataFrame(evaluation_rows)\n    actual = evaluation["thuc_te"]\n\n    ai_mae = mean_absolute_error(actual, evaluation["du_bao_ai"])\n    base_mae = mean_absolute_error(actual, evaluation["du_bao_tuan_truoc"])\n    ai_rmse = float(np.sqrt(\n        mean_squared_error(actual, evaluation["du_bao_ai"])\n    ))\n\n    print("MAE AI:", round(ai_mae, 3), "don/ngay")\n    print("MAE TUAN TRUOC:", round(base_mae, 3), "don/ngay")\n    print("RMSE AI:", round(ai_rmse, 3), "don/ngay")\n\n    if ai_mae < base_mae:\n        print("KET QUA: AI co MAE thap hon baseline tren lan thu nay.")\n    elif ai_mae > base_mae:\n        print("KET QUA: AI chua tot hon baseline tren lan thu nay.")\n    else:\n        print("KET QUA: AI va baseline co MAE bang nhau.")\n\n    print("3. HOC LAI TOAN BO DU LIEU SAU DANH GIA")\n    final_model = fit_model(y, dates, first_day)\n    next_day = dates[-1] + timedelta(days=1)\n    future = forecast(final_model, y, next_day, first_day, HORIZON)\n\n    forecast_rows = [\n        {\n            "ngay": (next_day + timedelta(days=i)).isoformat(),\n            "so_don_du_kien": int(round(pred)),\n            "nguon_du_lieu": source, "run_id": run_id,\n        }\n        for i, pred in enumerate(future)\n    ]\n\n    evaluation.to_csv(\n        output / "kiem_tra_du_bao.csv",\n        index=False, encoding="utf-8-sig",\n    )\n    pd.DataFrame(forecast_rows).to_csv(\n        output / "du_bao_7_ngay.csv",\n        index=False, encoding="utf-8-sig",\n    )\n\n    report = {\n        "schema_version": 2,\n        "run_id": run_id,\n        "data_source": source,\n        "timezone": "Asia/Ho_Chi_Minh",\n        "reference_date": reference_date.isoformat(),\n        "training_days": len(y),\n        "input_sha256": sha256(Path(input_path).read_bytes()),\n        "target": "so_don_theo_ngay",\n        "seed": SEED,\n        "horizon_days": HORIZON,\n        "evaluation_days": TEST_DAYS,\n        "evaluation_method": "4 rolling origins, retrain before each 7-day forecast",\n        "mae_ai": float(ai_mae),\n        "mae_baseline": float(base_mae),\n        "rmse_ai": ai_rmse,\n        "metrics_use_unrounded_predictions": True,\n        "final_model_refitted_on_all_days": True,\n        "training_last_date": dates[-1].isoformat(),\n        "sklearn_version": sklearn.__version__,\n        "limitation": ("Du lieu mo phong; chua chung minh hieu qua thuc te." if source == "mo_phong" else "Can danh gia them tren du lieu ban hang thuc te."),\n    }\n\n    joblib.dump(\n        {\n            "schema_version": 1,\n            "model": final_model,\n            "features": FEATURES,\n            "first_day": first_day.isoformat(),\n            "last_day": dates[-1].isoformat(),\n            "last_28_orders": y[-28:].tolist(),\n            "metadata": report,\n        },\n        output / "forecast_demo.joblib",\n    )\n    # Bind the opaque model artifact to the JSON metadata without importing\n    # joblib/sklearn in either the publisher or the web process.\n    report["model_sha256"] = sha256((output / "forecast_demo.joblib").read_bytes())\n    (output / "evaluation.json").write_text(\n        json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False),\n        encoding="utf-8",\n    )\n\n    print("4. DU BAO 7 NGAY TIEP THEO:")\n    for row in forecast_rows:\n        print(row["ngay"], ":", row["so_don_du_kien"], "don")\n\n    print("DA LUU KET QUA:", output.resolve())\n    print("NGUON:", source)\n', 'forecast_notebook.py': '"""Offline notebook preparation/reporting. Training uses the unchanged forecast pipeline."""\nimport argparse\nfrom datetime import date, datetime, timedelta, timezone\nfrom importlib import metadata\nimport json\nfrom pathlib import Path\nimport platform\nimport sys\nfrom uuid import uuid4\nimport zipfile\n\nfrom .forecast_demo import FIELDS, extend_demo\nfrom .forecast_store import (SOURCES, atomic_bytes, atomic_json, csv_bytes, load_bundle,\n                             read_daily, sha256, training_rows, vietnam_today)\n\nPLOTS = ("01_lich_su.png", "02_kiem_tra_28_ngay.png", "03_mae_theo_buoc.png",\n         "04_du_bao_7_ngay.png", "05_dac_trung.png")\n\n\ndef prepare_run(*, root, mode="demo", source="mo_phong", reference_date=None, csv_path=None):\n    if mode not in ("demo", "csv") or source not in SOURCES:\n        raise ValueError("Chế độ phải là demo/csv; nguồn phải là mo_phong/application_export.")\n    if mode == "demo" and source != "mo_phong":\n        raise ValueError("Không sinh dữ liệu mô phỏng cho nguồn application_export. Hãy chọn csv.")\n    reference = date.fromisoformat(reference_date) if reference_date else vietnam_today()\n    if mode == "csv" and (not csv_path or not Path(csv_path).is_file()):\n        raise FileNotFoundError("Chưa có CSV. Hãy tải đúng một CSV hoặc đặt CSV_PATH; không chuyển sang demo.")\n    run_id = "notebook_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_") + uuid4().hex[:12]\n    folder = Path(root).resolve() / run_id\n    folder.mkdir(parents=True, exist_ok=False)\n    original = Path(csv_path) if mode == "csv" else folder / "demo_input.csv"\n    if mode == "demo":\n        extend_demo(original, reference - timedelta(days=1))\n    all_rows = read_daily(original, source)\n    rows = training_rows(original, source, reference)\n    # Export only the three public aggregate fields, never extra uploaded columns.\n    snapshot = [{key: row[key] for key in FIELDS} for row in rows]\n    atomic_bytes(folder / "input.csv", csv_bytes(snapshot, FIELDS))\n    dataset = {\n        "run_id": run_id, "data_source": source, "data_mode": mode,\n        "reference_date": reference.isoformat(), "timezone": "Asia/Ho_Chi_Minh",\n        "training_days": len(rows), "training_first_date": rows[0]["ngay"],\n        "training_last_date": rows[-1]["ngay"], "uploaded_rows": len(all_rows),\n        "excluded_future_rows": len(all_rows) - len(rows), "source_sha256": sha256(original.read_bytes()),\n        "input_sha256": sha256((folder / "input.csv").read_bytes()),\n        "evaluation_first_date": (reference - timedelta(days=28)).isoformat(),\n        "limitation": ("Dữ liệu mô phỏng, chưa chứng minh chất lượng trên bán hàng thực tế."\n                       if source == "mo_phong" else\n                       "Nguồn CSV ứng dụng do người dùng cung cấp; có thể chứa đơn thử, chưa xác minh dữ liệu kinh doanh thực tế."),\n    }\n    atomic_json(folder / "dataset.json", dataset)\n    return folder\n\n\ndef read_dataset(folder):\n    folder = Path(folder)\n    dataset = json.loads((folder / "dataset.json").read_text(encoding="utf-8"))\n    if dataset["run_id"] != folder.name:\n        raise ValueError("Run ID không khớp thư mục notebook.")\n    if dataset["input_sha256"] != sha256((folder / "input.csv").read_bytes()):\n        raise ValueError("CSV đầu vào đã thay đổi sau bước kiểm tra.")\n    rows = training_rows(folder / "input.csv", dataset["data_source"], date.fromisoformat(dataset["reference_date"]))\n    if len(rows) != dataset["training_days"]:\n        raise ValueError("Số ngày không khớp metadata.")\n    return dataset\n\n\ndef train_run(folder):\n    from .forecast_training import train\n    folder = Path(folder)\n    dataset = read_dataset(folder)\n    train(folder / "input.csv", folder, dataset["data_source"],\n          date.fromisoformat(dataset["reference_date"]), dataset["run_id"])\n    load_bundle(folder, dataset["data_source"], date.fromisoformat(dataset["reference_date"]), dataset["run_id"], full=True)\n\n\ndef create_report(folder):\n    import joblib\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.dates as mdates\n    import matplotlib.pyplot as plt\n    import numpy as np\n    import pandas as pd\n    from .forecast_training import FEATURES\n\n    folder = Path(folder)\n    if (folder / "notebook_report.json").exists():\n        raise FileExistsError("Báo cáo đã tồn tại. Chạy lại từ bước tạo run để giữ lịch sử.")\n    dataset = read_dataset(folder)\n    report, _ = load_bundle(folder, dataset["data_source"], date.fromisoformat(dataset["reference_date"]),\n                            dataset["run_id"], full=True)\n    history = pd.read_csv(folder / "input.csv", parse_dates=["ngay"])\n    evaluation = pd.read_csv(folder / "kiem_tra_du_bao.csv", parse_dates=["ngay", "ngay_bat_dau_du_bao"])\n    future = pd.read_csv(folder / "du_bao_7_ngay.csv", parse_dates=["ngay"])\n    evaluation["sai_so_ai"] = abs(evaluation["thuc_te"] - evaluation["du_bao_ai"])\n    evaluation["sai_so_baseline"] = abs(evaluation["thuc_te"] - evaluation["du_bao_tuan_truoc"])\n    by_horizon = evaluation.groupby("buoc_du_bao", as_index=False).agg(\n        mae_ai=("sai_so_ai", "mean"), mae_baseline=("sai_so_baseline", "mean"),\n    )\n    by_origin = evaluation.groupby("ngay_bat_dau_du_bao", as_index=False).agg(\n        mae_ai=("sai_so_ai", "mean"), mae_baseline=("sai_so_baseline", "mean"), so_ngay=("ngay", "size"),\n    )\n    by_horizon.to_csv(folder / "mae_theo_buoc.csv", index=False, encoding="utf-8-sig")\n    by_origin.to_csv(folder / "mae_theo_luot.csv", index=False, date_format="%Y-%m-%d", encoding="utf-8-sig")\n    # This file was trained in this run and validated above; no uploaded joblib is loaded.\n    payload = joblib.load(folder / "forecast_demo.joblib")\n    model = payload["model"]\n    importance = pd.DataFrame({"dac_trung": FEATURES, "muc_quan_trong": model.feature_importances_})\n    importance = importance.sort_values("muc_quan_trong", ascending=False)\n    importance.to_csv(folder / "feature_importance.csv", index=False, encoding="utf-8-sig")\n    plots = folder / "charts"\n    plots.mkdir(exist_ok=True)\n    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.spines.top": False,\n                         "axes.spines.right": False, "axes.titlesize": 13, "figure.facecolor": "white"})\n    ai_color, base_color, actual_color = "#9d1231", "#ba862d", "#254e42"\n    source_label = "Dữ liệu mô phỏng" if dataset["data_source"] == "mo_phong" else "CSV ứng dụng — nguồn chưa xác minh"\n\n    def save(fig, ax, name, dated=True):\n        ax.set_ylabel("Số đơn / ngày")\n        ax.set_ylim(bottom=0)\n        ax.grid(axis="y", alpha=0.18)\n        ax.set_axisbelow(True)\n        if dated:\n            ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m/%Y"))\n            fig.autofmt_xdate(rotation=25)\n        fig.text(0.01, 0.01, source_label, color="#666666", fontsize=9)\n        fig.tight_layout(rect=(0, .045, 1, 1))\n        fig.savefig(plots / name, dpi=160)\n        plt.close(fig)\n\n    fig, ax = plt.subplots(figsize=(12, 4.5))\n    ax.plot(history["ngay"], history["so_don"], color=actual_color, linewidth=1.1)\n    ax.axvspan(evaluation["ngay"].min(), evaluation["ngay"].max(), color=base_color, alpha=.16, label="28 ngày đánh giá")\n    ax.set_title("Lịch sử số đơn trước ngày tham chiếu")\n    ax.legend()\n    save(fig, ax, PLOTS[0])\n\n    fig, ax = plt.subplots(figsize=(12, 4.8))\n    for column, label, color in (("thuc_te", "Thực tế", actual_color), ("du_bao_ai", "Random Forest", ai_color),\n                                  ("du_bao_tuan_truoc", "Cùng thứ tuần trước", base_color)):\n        ax.plot(evaluation["ngay"], evaluation[column], marker="o", markersize=3, label=label, color=color)\n    for start in evaluation["ngay_bat_dau_du_bao"].drop_duplicates().iloc[1:]:\n        ax.axvline(start, color="#aaaaaa", linewidth=.8, linestyle=":")\n    ax.set_title("4 lượt dự báo 7 ngày trên 28 ngày cuối — đánh giá cuốn chiếu")\n    ax.legend(ncols=3)\n    save(fig, ax, PLOTS[1])\n\n    fig, ax = plt.subplots(figsize=(10, 4.5))\n    ax.bar(by_horizon["buoc_du_bao"] - .18, by_horizon["mae_ai"], .36, color=ai_color, label="Random Forest")\n    ax.bar(by_horizon["buoc_du_bao"] + .18, by_horizon["mae_baseline"], .36, color=base_color, label="Cùng thứ tuần trước")\n    ax.set_title("MAE theo bước dự báo — mỗi bước có 4 lần kiểm tra")\n    ax.set_xlabel("Bước dự báo (ngày)")\n    ax.set_xticks(range(1, 8))\n    ax.legend()\n    save(fig, ax, PLOTS[2], dated=False)\n\n    fig, ax = plt.subplots(figsize=(12, 4.8))\n    tail = history.tail(28)\n    ax.plot(tail["ngay"], tail["so_don"], marker="o", markersize=3, color=actual_color, label="Lịch sử 28 ngày")\n    ax.plot(future["ngay"], future["so_don_du_kien"], marker="o", linestyle="--", color=ai_color, label="Dự báo 7 ngày")\n    ax.axvline(future["ngay"].iloc[0], color="#aaaaaa", linestyle=":")\n    ax.set_title("Dự báo số đơn 7 ngày sau khi học lại toàn bộ lịch sử")\n    ax.legend()\n    save(fig, ax, PLOTS[3])\n\n    fig, ax = plt.subplots(figsize=(10, 5))\n    ax.barh(importance["dac_trung"][::-1], importance["muc_quan_trong"][::-1], color=ai_color)\n    ax.set_title("Mức quan trọng đặc trưng trong mô hình cuối (MDI)")\n    ax.set_xlabel("Tỷ trọng giảm độ hỗn tạp; không phải quan hệ nhân quả")\n    ax.grid(axis="x", alpha=.18)\n    fig.text(.01, .01, source_label, color="#666666", fontsize=9)\n    fig.tight_layout(rect=(0, .045, 1, 1))\n    fig.savefig(plots / PLOTS[4], dpi=160)\n    plt.close(fig)\n\n    base_rmse = float(np.sqrt(np.mean((evaluation["thuc_te"] - evaluation["du_bao_tuan_truoc"]) ** 2)))\n    improvement = ((report["mae_baseline"] - report["mae_ai"]) / report["mae_baseline"] * 100\n                   if report["mae_baseline"] else None)\n    result = {\n        **report, "rmse_baseline": base_rmse, "mae_improvement_percent": improvement,\n        "model_parameters": model.get_params(), "feature_names": FEATURES,\n        "versions": {name: metadata.version(name) for name in ("numpy", "pandas", "scikit-learn", "joblib", "matplotlib", "tzdata")},\n        "python_version": platform.python_version(), "charts": list(PLOTS),\n        "excluded_future_rows": dataset["excluded_future_rows"],\n        "conclusion": ("MAE Random Forest thấp hơn baseline trong phép kiểm tra này."\n                       if report["mae_ai"] < report["mae_baseline"] else\n                       "Random Forest chưa có MAE thấp hơn baseline trong phép kiểm tra này."),\n        "limitations": [dataset["limitation"],\n                        "Không dùng kết quả đánh giá này để tự chọn lại tham số rồi báo cùng tập là kiểm tra độc lập.",\n                        "Dự báo nhiều ngày dùng lại dự báo các bước trước; không dùng số thực tế tương lai.",\n                        "Feature importance MDI chỉ mô tả mô hình cuối trên train, không chứng minh quan hệ nhân quả.",\n                        "Notebook tạo kết quả offline; không tự công bố lên Dashboard hoặc thay đổi hosting."],\n    }\n    atomic_json(folder / "notebook_report.json", result)\n    (folder / "README_KET_QUA.md").write_text(\n        f"# Kết quả Random Forest QLNH\\n\\nRun: {dataset[\'run_id\']}\\n\\nNguồn: {dataset[\'data_source\']}\\n\\n"\n        f"{dataset[\'limitation\']}\\n\\nDữ liệu học đến {dataset[\'training_last_date\']}; "\n        f"dự báo bắt đầu {dataset[\'reference_date\']}.\\n\\n{result[\'conclusion\']}\\n\\n"\n        "Không tự đưa file này lên Dashboard. Model joblib chỉ nên mở khi tin cậy nguồn; "\n        "cần môi trường thư viện tương thích ghi trong notebook_report.json.\\n",\n        encoding="utf-8",\n    )\n    return result\n\n\ndef create_archive(folder):\n    folder = Path(folder)\n    dataset = read_dataset(folder)\n    load_bundle(folder, dataset["data_source"], date.fromisoformat(dataset["reference_date"]), dataset["run_id"], full=True)\n    report = json.loads((folder / "notebook_report.json").read_text(encoding="utf-8"))\n    if report["run_id"] != dataset["run_id"] or report["data_source"] != dataset["data_source"]:\n        raise ValueError("Báo cáo và dữ liệu không thuộc cùng lần chạy.")\n    names = ["input.csv", "dataset.json", "forecast_demo.joblib", "evaluation.json", "du_bao_7_ngay.csv",\n             "kiem_tra_du_bao.csv", "mae_theo_buoc.csv", "mae_theo_luot.csv", "feature_importance.csv",\n             "notebook_report.json", "README_KET_QUA.md", *[f"charts/{name}" for name in PLOTS]]\n    hashes = {name: sha256((folder / name).read_bytes()) for name in names}\n    manifest = {"run_id": dataset["run_id"], "data_source": dataset["data_source"], "sha256": hashes}\n    archive = folder.parent / (folder.name + "_" + uuid4().hex[:8] + ".zip")\n    with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as stream:\n        for name in names:\n            stream.write(folder / name, name)\n        stream.writestr("manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False))\n    return archive\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("action", choices=("train", "report"))\n    parser.add_argument("folder", type=Path)\n    args = parser.parse_args()\n    if args.action == "train":\n        train_run(args.folder)\n    else:\n        create_report(args.folder)\n\n\nif __name__ == "__main__":\n    sys.stdout.reconfigure(encoding="utf-8")\n    main()\n'}
for name, content in SOURCE_FILES.items():
    (PACKAGE / name).write_text(content, encoding="utf-8")
for name in list(sys.modules):
    if name == "ai_tools" or name.startswith("ai_tools."):
        del sys.modules[name]
sys.path.insert(0, str(SUPPORT_ROOT))
from ai_tools.forecast_notebook import prepare_run, create_archive

def run_worker(action):
    env = dict(os.environ, PYTHONPATH=str(SUPPORT_ROOT), PYTHONIOENCODING="utf-8",
               MPLBACKEND="Agg", MPLCONFIGDIR=str(WORK_ROOT / ".matplotlib"))
    process = subprocess.run([sys.executable, "-m", "ai_tools.forecast_notebook", action, str(OUTPUT_DIR)],
                             cwd=SUPPORT_ROOT, env=env, capture_output=True, text=True,
                             encoding="utf-8", errors="replace")
    if process.stdout:
        print(process.stdout)
    if process.returncode:
        raise RuntimeError("Bước " + action + " thất bại:\n" + process.stderr)
    if process.stderr:
        print(process.stderr)

def show_csv(path, limit=10):
    with Path(path).open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        fields = reader.fieldnames
        rows = [row for index, row in enumerate(reader) if index < limit]
    if HTML:
        header = "".join("<th>" + escape(name) + "</th>" for name in fields)
        body = "".join("<tr>" + "".join("<td>" + escape(str(row[name])) + "</td>" for name in fields) + "</tr>" for row in rows)
        display(HTML('<div style="overflow:auto"><table style="border-collapse:collapse" border="1" cellpadding="7"><thead><tr>' + header + '</tr></thead><tbody>' + body + '</tbody></table></div>'))
    else:
        print(rows)

def show_chart(name):
    path = OUTPUT_DIR / "charts" / name
    if Image:
        display(Image(filename=str(path)))
    else:
        print("Biểu đồ:", path)

print("Đã chuẩn bị thư viện và mã nguồn. Thư mục làm việc:", WORK_ROOT)

## 3. Đọc CSV, kiểm tra và tạo snapshot riêng

Mỗi lần chạy bước này tạo một run mới, giữ nguyên kết quả cũ. `input.csv` chỉ gồm 3 cột tổng hợp,
không đưa các cột phụ trong CSV tải lên vào ZIP kết quả. Không đọc hoặc ghi database nghiệp vụ.

In [ ]:
input_path = CSV_PATH.strip() or None
if DATA_MODE == "csv" and input_path is None:
    if colab_files is None:
        raise FileNotFoundError("Chạy ngoài Colab: cần đặt CSV_PATH đến file CSV có sẵn.")
    uploaded = colab_files.upload()
    if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith(".csv"):
        raise ValueError("Hãy chọn đúng một file CSV số đơn theo ngày.")
    input_path = next(iter(uploaded))

OUTPUT_DIR = prepare_run(root=WORK_ROOT, mode=DATA_MODE, source=DATA_SOURCE,
                         reference_date=REFERENCE_DATE.strip() or None, csv_path=input_path)
DATASET = json.loads((OUTPUT_DIR / "dataset.json").read_text(encoding="utf-8"))
print("Nguồn:", DATASET["data_source"], "| Run:", DATASET["run_id"])
print("Dữ liệu học:", DATASET["training_first_date"], "đến", DATASET["training_last_date"], "|", DATASET["training_days"], "ngày")
print("Dòng từ ngày tham chiếu trở đi đã loại:", DATASET["excluded_future_rows"])
print(DATASET["limitation"])
show_csv(OUTPUT_DIR / "input.csv", limit=7)

## 4. Đặc trưng và mô hình được sử dụng

| Đặc trưng | Ý nghĩa |
|---|---|
| `thu`, `cuoi_tuan`, `chi_so_ngay` | Thứ trong tuần, cuối tuần, số ngày tính từ mốc đầu |
| `don_hom_qua` | Số đơn ngày trước |
| `don_7_ngay_truoc`, `don_14_ngay_truoc`, `don_28_ngay_truoc` | Số đơn cùng ngày cách 7/14/28 ngày |
| `trung_binh_7_ngay`, `trung_binh_28_ngay` | Trung bình **các ngày trước**, không chứa ngày cần dự báo |

Random Forest: **200 cây**, `max_depth=8`, `min_samples_leaf=3`, `random_state=42`.
Mã dưới đây lấy từ cùng file nguồn được nhúng và thực thi trong notebook:

```python
def feature_row(day, history, first_day):
    if len(history) < 28:
        raise ValueError("Can 28 ngay lich su.")
    return [
        day.weekday(),
        int(day.weekday() >= 5),
        (day - first_day).days,
        history[-1],
        history[-7],
        history[-14],
        history[-28],
        float(np.mean(history[-7:])),
        float(np.mean(history[-28:])),
    ]

def fit_model(values, known_dates, first_day):
    # Moi dong chi dung lich su truoc ngay can du bao.
    X = [
        feature_row(known_dates[i], values[:i], first_day)
        for i in range(28, len(values))
    ]
    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=3,
        random_state=SEED,
        n_jobs=-1,
    )
    model.fit(pd.DataFrame(X, columns=FEATURES), values[28:])
    return model
```

**Dự báo nhiều bước:** ngày thứ hai dùng kết quả dự báo ngày thứ nhất cho giá trị trễ chưa có thực tế;
không lấy dữ liệu thực tế trong 7 ngày kiểm tra để đưa vào đặc trưng giữa lượt dự báo.

**Đánh giá:** giữ 28 ngày cuối, chia 4 lượt liên tiếp × 7 ngày. Mỗi lượt học lại chỉ từ dữ liệu trước lượt đó.
Baseline dùng số đơn cùng thứ tuần trước. Sau đánh giá mới học lại toàn bộ dữ liệu để dự báo 7 ngày mới.
Không chia ngẫu nhiên chuỗi thời gian và không chọn lại tham số dựa trên cùng 28 ngày rồi coi đó là kiểm tra độc lập.

In [ ]:
run_worker("train")
EVALUATION = json.loads((OUTPUT_DIR / "evaluation.json").read_text(encoding="utf-8"))
print("MAE Random Forest:", round(EVALUATION["mae_ai"], 3), "đơn/ngày")
print("MAE baseline:", round(EVALUATION["mae_baseline"], 3), "đơn/ngày")
print("RMSE Random Forest:", round(EVALUATION["rmse_ai"], 3), "đơn/ngày")

## 5. Đánh giá và biểu đồ

MAE là sai số tuyệt đối trung bình, tính theo đơn/ngày; RMSE làm nổi rõ hơn sai số lớn.
Các chỉ số dùng dự báo chưa làm tròn. Kết quả không bảo đảm Random Forest tốt hơn baseline.
Vạch dọc trên biểu đồ đánh giá chia 4 lượt dự báo. Không có khoảng tin cậy vì pipeline chưa ước lượng đại lượng đó.

In [ ]:
run_worker("report")
REPORT = json.loads((OUTPUT_DIR / "notebook_report.json").read_text(encoding="utf-8"))
print(REPORT["conclusion"])
print("RMSE baseline:", round(REPORT["rmse_baseline"], 3), "đơn/ngày")
if REPORT["mae_improvement_percent"] is not None:
    print("Mức giảm MAE so với baseline:", round(REPORT["mae_improvement_percent"], 2), "% (âm nghĩa là kém hơn)")
else:
    print("Baseline có MAE = 0; không chia cho 0 để tính phần trăm cải thiện.")
show_csv(OUTPUT_DIR / "mae_theo_luot.csv")
show_csv(OUTPUT_DIR / "mae_theo_buoc.csv")
show_chart("01_lich_su.png")
show_chart("02_kiem_tra_28_ngay.png")
show_chart("03_mae_theo_buoc.png")

## 6. Dự báo 7 ngày tiếp theo và giải thích đặc trưng

Bảng dưới là dự báo mới sau khi học lại toàn bộ snapshot; số đơn nguyên, không âm.
Biểu đồ feature importance MDI mô tả mô hình cuối, không chứng minh nguyên nhân hoặc độ chính xác ngoài dữ liệu học.
Không gọi những đặc trưng có điểm cao là nguyên nhân làm tăng số đơn.

In [ ]:
show_csv(OUTPUT_DIR / "du_bao_7_ngay.csv")
show_chart("04_du_bao_7_ngay.png")
show_csv(OUTPUT_DIR / "feature_importance.csv")
show_chart("05_dac_trung.png")
for limitation in REPORT["limitations"]:
    print("•", limitation)

## 7. Tải bộ kết quả

ZIP chứa snapshot CSV, model `forecast_demo.joblib`, dự báo 7 ngày, 28 ngày đánh giá,
MAE theo lượt/bước, feature importance, 5 biểu đồ PNG, metadata, phiên bản thư viện và manifest checksum.
Tên model được giữ tương thích mã hiện có; hãy đọc `data_source` trong metadata để xác định nguồn.
Chỉ mở model joblib từ nguồn tin cậy và dùng môi trường thư viện tương thích.

Notebook **không công bố lên Dashboard**. Web chỉ đọc các run được updater kiểm tra và công bố riêng.
Không chép đè từng file vào bộ đang chạy trên website. CSV mô phỏng và CSV ứng dụng vẫn là hai nguồn tách biệt.

In [ ]:
ARCHIVE = create_archive(OUTPUT_DIR)
print("Đã lưu run:", OUTPUT_DIR)
print("ZIP:", ARCHIVE)
if DOWNLOAD_RESULTS and colab_files is not None:
    colab_files.download(str(ARCHIVE))
elif DOWNLOAD_RESULTS:
    print("Chạy local: mở file ZIP ở đường dẫn trên.")

## Tài liệu và giới hạn

- [Colab: mở/tải notebook](https://research.google.com/colaboratory/faq.html)
- [Colab: upload/download file](https://github.com/googlecolab/colabtools/blob/main/google/colab/files.py)
- [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)
- [Giới hạn feature importance dựa trên impurity](https://scikit-learn.org/stable/auto_examples/inspection/plot_permutation_importance.html)

Đây là thí nghiệm tái lập quy trình dự báo đang có. Muốn đánh giá hiệu quả nhà hàng thực tế cần dữ liệu vận hành được xác minh,
nhiều giai đoạn kiểm tra và đối chiếu nhu cầu thực tế; không suy ra chất lượng kinh doanh từ dữ liệu mô phỏng.